In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from sklearn.datasets import make_regression

In [3]:
X, y = make_regression(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    noise=10,
    random_state=42
)

# Convert to DataFrame
df = pd.DataFrame(X, columns=["Feature_1", "Feature_2"])
df["Target"] = y

# Display first few rows
print(df.head())

   Feature_1  Feature_2     Target
0  -0.167118   0.146714 -14.996950
1  -0.020902   0.117327 -12.678089
2   0.150419   0.364961  17.775455
3   0.555604   0.089581   6.661465
4   0.058209  -1.142970 -14.195530


In [4]:
# plotting the data
import plotly.express as px

# 3D Scatter Plot
fig = px.scatter_3d(
    df,
    x="Feature_1",
    y="Feature_2",
    z="Target",
    color="Target",
    title="Synthetic Regression Dataset",
    opacity=0.8
)

fig.update_layout(
    scene=dict(
        xaxis_title="Feature 1",
        yaxis_title="Feature 2",
        zaxis_title="Target"
    ),
    template="plotly_white"
)

fig.show()

In [27]:
class MultipleLinearRegressionOLS:
  def __init__(self):
    self.coef_ = None
    self.intercept_ = None

  def fit(self,X_train,y_train):
    X_train = np.insert(X_train,0,1,axis = 1)
    self.coef_ = np.linalg.inv(X_train.T @ X_train) @ X_train.T @ y_train
    self.intercept_ = self.coef_[0]
    self.coef_ = self.coef_[1:]
    print(f"Coef:- {self.coef_}")
    print(f"Intercept:- {self.intercept_}")

  def predict(self, X_test):
    return X_test.values @ self.coef_ + self.intercept_

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train,X_test,y_train,y_test = train_test_split(df.drop("Target",axis = 1),df["Target"],test_size = 0.2,random_state = 42)

In [11]:
X_train.shape

(800, 2)

In [28]:
mlr = MultipleLinearRegressionOLS()

In [29]:
mlr.fit(X_train,y_train)

Coef:- [41.39506319  6.6740871 ]
Intercept:- 0.11251608309309952


In [30]:
from sklearn.metrics import r2_score

In [31]:
y_pred = mlr.predict(X_test)

In [32]:
r2_score(y_true=y_test,y_pred=y_pred)

0.9362992668663301

# Comparing with sklearn

In [16]:
from sklearn.linear_model import LinearRegression

In [17]:
sk_mlr = LinearRegression()


In [18]:
sk_mlr.fit(X_train,y_train)

LinearRegression()

In [19]:
sk_mlr.coef_

array([41.39506319,  6.6740871 ])

In [33]:
sk_mlr.intercept_

np.float64(0.1125160830931009)

# Plotting Hyperplane

In [35]:
import numpy as np
import plotly.graph_objects as go

# Scatter plot of the actual data
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=df["Feature_1"],
        y=df["Feature_2"],
        z=df["Target"],
        mode="markers",
        marker=dict(
            size=3,
            color=df["Target"],
            colorscale="Viridis",
            opacity=0.7
        ),
        name="Data"
    )
)

# Create a grid over the feature space
x1_range = np.linspace(df["Feature_1"].min(), df["Feature_1"].max(), 30)
x2_range = np.linspace(df["Feature_2"].min(), df["Feature_2"].max(), 30)

X1, X2 = np.meshgrid(x1_range, x2_range)

# Plane equation
Z = (
    mlr.coef_[0] * X1
    + mlr.coef_[1] * X2
    + mlr.intercept_
)

# Add regression plane
fig.add_trace(
    go.Surface(
        x=X1,
        y=X2,
        z=Z,
        opacity=0.5,
        colorscale="Reds",
        showscale=False,
        name="Regression Plane"
    )
)

fig.update_layout(
    title="Multiple Linear Regression Hyperplane",
    scene=dict(
        xaxis_title="Feature 1",
        yaxis_title="Feature 2",
        zaxis_title="Target"
    ),
    width=900,
    height=700
)

fig.show()